**[Course Home](../README.md) | [Python Lab](../python/05_curvature_laplace_and_fisher_information.ipynb) | [Prev: Ch 04: Shared Outages & Venn Geometry](04_mutual_information_and_venn_geometry.ipynb) | [Next: Ch 06: The Engine of Computation: MCMC](06_mcmc_mechanics_and_entropy.ipynb)**

*Part II: Multi-Variable Geometry & Causal Architectures — Chapter 05 of 12*

---

# Chapter 05: The Geometry of Inference
### *Laplace Curvature, The Hessian Matrix, The Cramér-Rao Bound, and Fisher Information Geometry*

---

In Chapters 01–04, inference lived largely in low-dimensional conjugate or
discrete models: updating a single flaky test rate $\theta$ with an exact
Beta-Binomial formula, or measuring shared outages across discrete events. In
real production telemetry, however, a test's failure probability depends on
CPU load, concurrency, and latency simultaneously—breaking conjugacy and
leaving a multi-parameter posterior with no closed form.

To navigate these intractable multi-parameter spaces, Chapter 05 develops a
unified geometric story in four steps:

1. **Local Gaussian Approximation (Laplace, Sec 1–2):** When the exact
   posterior has no closed form, find its peak and match its downward
   curvature with a multivariate Gaussian.
2. **Curvature as Certainty (Hessian, Sec 3):** Read parameter error bars and
   correlated trade-offs directly from the Hessian matrix $\mathbf{H}$.
3. **Expected Curvature & Limits (Fisher, Sec 4):** Shift from one observed
   dataset ($\mathbf{H}$) to what any experiment can possibly reveal
   ($\mathcal{I}$)—unlocking Cramér-Rao precision floors and
   coordinate-invariant Jeffreys priors.
4. **Curved Manifolds & Action (Information Geometry, Sec 5–6):** Measure true
   probability distance in bits rather than raw parameter coordinates, enabling
   Natural Gradient Descent, dynamic timeout sizing, and multicollinearity
   alarms.

---

## 1. When Conjugacy Dies: Multi-Parameter Production Telemetry

In Chapter 01, we updated a 1-dimensional failure rate $\theta$ using the
Beta-Binomial conjugate engine. Because the model was 1D and conjugate, the
math reduced to elementary addition: $\alpha + k, \beta + (N - k)$.

Now, step into a real production backend system.

A checkout integration test does not fail in a vacuum. Its probability of
failure depends on **multiple interacting continuous covariates** (measured
input variables):
* $x_1$: CPU utilization of the test runner ($0\%$ to $100\%$)
* $x_2$: Number of concurrent builds on the CI worker cluster ($1$ to $64$)
* $x_3$: Network latency to the mock payment gateway in milliseconds
  ($10\text{ ms}$ to $2,000\text{ ms}$)

Why not just add up the risk from each metric directly? Suppose you try a
simple straight-line rule: *"Every $10\%$ of CPU load adds $+1\%$ failure risk,
and every $100\text{ ms}$ of network latency adds $+20\%$ failure risk."* That
works fine at normal load ($50\%$ CPU and $100\text{ ms}$ latency gives
$5\% + 20\% = 25\%$ failure risk). **But during an outage** ($90\%$ CPU and
$600\text{ ms}$ latency), adding them directly gives
$9\% + 120\% = \mathbf{129\%}$ failure probability—which is physically
impossible because a probability can never exceed $100\%$ (or drop below
$0\%$). Switching to **log-odds** (`logit`, from Chapter 00) solves this:
stretching $[0, 1]$ onto $(-\infty, +\infty)$ lets us add risk factors cleanly
without hitting $0\%$ or $100\%$ walls, and inverting back via the sigmoid
squashes any total into a valid probability.

Assigning a weight $\beta_i$ to each covariate gives a multi-variable logistic
regression, where the **logit** function
$\text{logit}(\theta) = \ln \frac{\theta}{1-\theta}$ maps a failure
probability $\theta \in (0, 1)$ to unbounded log-odds on
$(-\infty, +\infty)$:

$$
\text{logit}(P(\text{Fail} \mid \mathbf{x})) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3
$$

Now, our unknown parameter vector is **4-dimensional**:

$$
\boldsymbol{\theta} = (\beta_0, \beta_1, \beta_2, \beta_3) \in \mathbb{R}^4
$$

**Why Conjugacy Breaks Here**:
Beta priors and Binomial likelihoods are conjugate because both share the same
$\theta^a (1 - \theta)^b$ algebraic template, so multiplying them simply adds
their exponents. In logistic regression, however, multiplying a Gaussian prior
over $\boldsymbol{\beta}$ by a product of logistic likelihood terms
$1 / (1 + e^{-\mathbf{x}_i^T \boldsymbol{\beta}})$ mixes incompatible
functional forms. As a result, the joint posterior distribution:

$$
P(\boldsymbol{\theta} \mid D) \propto P(\boldsymbol{\theta}) \prod_{i=1}^N P(y_i \mid \mathbf{x}_i, \boldsymbol{\theta})
$$

is **intractable** (it has no closed-form integral), forming a non-standard
4-dimensional surface you cannot integrate on paper.

How do we estimate $\boldsymbol{\theta}$? How do we measure our remaining
uncertainty? And what is the **geometry** of this probability space?

---

## 2. The Laplace Approximation: Finding the Peak

When exact global integration is impossible, we replace the full 4-dimensional
surface with a local quadratic model anchored at its peak. In 1774,
Pierre-Simon Laplace proposed a practical insight:
> *Almost any smooth probability distribution, when magnified around its
> highest peak, looks nearly indistinguishable from a **multivariate Gaussian
> distribution**.*

Why do we approximate the **log-posterior** $\ln P(\boldsymbol{\theta} \mid D)$
instead of $P(\boldsymbol{\theta} \mid D)$ directly? Taking the logarithm
converts a curved Gaussian bell $\exp(-\frac{1}{2}(x-\mu)^2/\sigma^2)$ into a
simple downward-opening quadratic parabola, and turns massive likelihood
products across $N$ test runs into numerically stable sums.

To visualize this quadratic behavior before tackling multi-dimensional
problems, consider approximating a 1-D $\text{Beta}(2, 10)$ log-posterior
around its mode $\hat{\theta} = 0.10$ with curvature $H = 111.11$:

![The Laplace Approximation Around the Mode: Quadratic Taylor Parabola](../images/ch05_laplace_tangent.png)

### Step 1: Find the Peak (MAP)
We locate the parameter vector $\hat{\boldsymbol{\theta}}$ that maximizes the
log-posterior—the **MAP** (Maximum A Posteriori, the posterior mode):

$$
\hat{\boldsymbol{\theta}} = \arg\max_{\boldsymbol{\theta}} \ln P(\boldsymbol{\theta} \mid D)
$$

At the summit of a smooth hill, every direction slopes downward, so the local
tangent plane is flat and the gradient vector is exactly zero:

$$
\nabla \ln P(\hat{\boldsymbol{\theta}} \mid D) = \mathbf{0}
$$

### Step 2: Taylor Expand the Log-Posterior
We perform a second-order Taylor series expansion of
$\ln P(\boldsymbol{\theta} \mid D)$ around the peak
$\hat{\boldsymbol{\theta}}$:

$$
\ln P(\boldsymbol{\theta} \mid D) \approx \ln P(\hat{\boldsymbol{\theta}} \mid D) + \underbrace{\nabla \ln P(\hat{\boldsymbol{\theta}} \mid D)^T (\boldsymbol{\theta} - \hat{\boldsymbol{\theta}})}_{= \mathbf{0} \text{ at the peak!}} - \frac{1}{2} (\boldsymbol{\theta} - \hat{\boldsymbol{\theta}})^T \mathbf{H} (\boldsymbol{\theta} - \hat{\boldsymbol{\theta}})
$$

where $\mathbf{H}$ is the **Hessian Matrix** of negative second derivatives
evaluated at the mode:

$$
\mathbf{H}_{ij} \equiv -\left. \frac{\partial^2 \ln P(\boldsymbol{\theta} \mid D)}{\partial \theta_i \partial \theta_j} \right|_{\hat{\boldsymbol{\theta}}}
$$

Why the minus sign? Because $\hat{\boldsymbol{\theta}}$ is a summit, the
log-posterior bends *downward*, making its second derivatives negative;
negating them makes $\mathbf{H}$ positive-definite so sharper peaks correspond
to larger positive precision numbers.

### Step 3: Exponentiate Both Sides
Exponentiating the quadratic approximation yields:

$$
P(\boldsymbol{\theta} \mid D) \approx C \cdot \exp\left( -\frac{1}{2} (\boldsymbol{\theta} - \hat{\boldsymbol{\theta}})^T \mathbf{H} (\boldsymbol{\theta} - \hat{\boldsymbol{\theta}}) \right)
$$

Here $C = P(\hat{\boldsymbol{\theta}} \mid D)$ is a constant height factor
that is absorbed into the Gaussian normalizing constant when we normalize the
density to integrate to $1$. This is precisely the formula for a
**Multivariate Normal Distribution**:

$$
\mathbf{P(\boldsymbol{\theta} \mid D) \approx \mathcal{N}\left( \hat{\boldsymbol{\theta}}, \; \mathbf{\Sigma} \right), \qquad \text{where } \mathbf{\Sigma} = \mathbf{H}^{-1}}
$$

> **The Key Result**: The Laplace approximation shows that the local covariance
> matrix (uncertainty) of a Bayesian model is simply the **inverse of the
> Hessian matrix** ($\mathbf{\Sigma} = \mathbf{H}^{-1}$).

### Quantifying Laplace Approximation Error on Small-Sample Flaky Tests

While the Laplace approximation works well asymptotically ($N \to \infty$),
how does it perform on small-sample telemetry typical of newly authored tests?

Consider a new integration test observed over $N = 10$ runs with $k = 1$
failure and $9$ passes. Under a flat prior $p(\theta) = \text{Beta}(1, 1)$, the
exact Bayesian posterior is conjugate:

$$
p(\theta \mid D) = \text{Beta}(1 + 1, \; 1 + 9) = \text{Beta}(2, 10)
$$

Because the $\text{Beta}(1, 1)$ prior is constant ($\ln 1 = 0$), the
log-posterior equals the Bernoulli log-likelihood up to an additive constant:

$$
\ln L(\theta) = \ln \theta + 9 \ln(1 - \theta)
$$

The maximum a posteriori (MAP) peak is:

$$
\hat{\theta}_{\text{MAP}} = \frac{1}{10} = 0.1000
$$

Evaluating the Hessian (negative second derivative) at $\hat{\theta}$:

$$
H = -\left. \frac{d^2 \ln L}{d\theta^2} \right|_{\hat{\theta}}
  = \frac{1}{\hat{\theta}^2} + \frac{9}{(1 - \hat{\theta})^2}
$$

$$
H = \frac{1}{(0.10)^2} + \frac{9}{(0.90)^2} = 100 + 11.1111 = 111.1111
$$

The Laplace Gaussian approximation has standard deviation:

$$
\sigma_{\text{Laplace}} = \frac{1}{\sqrt{H}} = \frac{1}{\sqrt{111.1111}} \approx 0.0949
$$

yielding the symmetric approximation $\mathcal{N}(0.1000, \; 0.0949^2)$.

Now compare the exact Beta posterior against the Gaussian Laplace fit:

| Metric | Exact Posterior $\text{Beta}(2, 10)$ | Gaussian Laplace $\mathcal{N}(0.1000, 0.0949^2)$ | Operational Consequence |
| :--- | :--- | :--- | :--- |
| **Mode** | $0.1000$ | $0.1000$ | Exact match at peak |
| **Mean** | $0.1667$ | $0.1000$ | $-40.0\%$ bias (ignores skew) |
| **Median** | $0.1480$ | $0.1000$ | $-32.4\%$ shift |
| **Std Dev ($\sigma$)** | $0.1034$ | $0.0949$ | Underestimates variance by $8.2\%$ |
| **95% Credible Interval** | $[0.0228, 0.4128]$ | $[-0.0860, 0.2860]$ | Unphysical lower bound; cuts tail |
| **$P(\theta < 0)$ (Impossible)** | $\mathbf{0.0000\%}$ | $\mathbf{14.59\%}$ | **1 in 7 draws has negative rate!** |

![Exact Beta(2, 10) vs. Gaussian Laplace Fit: Negative Spillover Breakdown](../images/ch05_exact_beta_vs_laplace.png)

**Why Does Laplace Fail on Small Samples?**
1. **Domain Mismatch**: A Gaussian lives on $(-\infty, +\infty)$, while
   failure rates $\theta$ live strictly on $[0, 1]$. Near the boundary
   $\theta \to 0$, the symmetric bell spills $14.59\%$ of its probability mass
   into negative territory ($\theta < 0$).
2. **Ignored Skewness**: Rare-event posteriors are heavily right-skewed. The
   exact posterior mean ($0.1667$) is $67\%$ higher than the mode ($0.1000$)
   because small samples cannot rule out a true failure rate of $30\%$ or
   $40\%$. The Laplace approximation treats the mode as the center of mass,
   underestimating flakiness risk.

**The Production Remedy**:
* **Reparameterize to Logit Space**: Fit Laplace to log-odds
  $\phi = \ln \frac{\theta}{1 - \theta} \in (-\infty, +\infty)$, guaranteeing
  $\theta \in (0, 1)$ after mapping back via the **sigmoid** function
  $\sigma(\phi) = \frac{1}{1 + e^{-\phi}}$.
* **Exact Conjugacy or MCMC**: When sample size $N \le 30$ or flake rate
  $\theta \le 0.05$, use exact Beta conjugate updates or MCMC sampling.

With our Gaussian approximation $\mathbf{\Sigma} = \mathbf{H}^{-1}$ in hand, let
us examine what the curvature matrix $\mathbf{H}$ tells us geometrically about
parameter certainty and correlated error bars.

---

## 3. The Hessian as Information: Curvature Equals Certainty

Why does curvature equal certainty? From the Taylor expansion, shifting a
parameter by $\Delta\theta$ drops the log-posterior by
$\Delta \ln P \approx -\frac{1}{2} H (\Delta\theta)^2$. Steep curvature makes
even tiny shifts drop the posterior sharply, whereas flat curvature leaves a
wide band of plausible values:

* **High Second Derivative ($\mathbf{H}$ is large)**:
  * The log-posterior curves sharply like a narrow canyon.
  * The inverse $\mathbf{\Sigma} = \mathbf{H}^{-1}$ is small.
  * **Physical Meaning**: The data pins down the parameter with high
    certainty.
* **Low Second Derivative ($\mathbf{H}$ is small)**:
  * The log-posterior is a broad, shallow saucer.
  * The inverse $\mathbf{\Sigma} = \mathbf{H}^{-1}$ is large.
  * **Physical Meaning**: The data barely constrains the parameter; high
    uncertainty lingers.

$$
\mathbf{\text{Curvature} \equiv \text{Information} \equiv \frac{1}{\text{Uncertainty}}}
$$

**Off-Diagonal Curvature and Correlated Ridges ($\mathbf{H}_{ij} \neq 0$)**:
In multi-parameter models, covariates often co-occur—for instance, high CPU
utilization ($x_1$) and high build concurrency ($x_2$) tend to spike together.
When that happens, the log-posterior forms a **tilted diagonal ridge** rather
than an axis-aligned bowl. Stepping $\beta_1$ up while stepping $\beta_2$ down
keeps the linear predictor $\beta_1 x_1 + \beta_2 x_2$ nearly unchanged, so the
curvature *along the ridge* is shallow even if each individual slice looks
steep. Inverting the coupled matrix ($\mathbf{\Sigma} = \mathbf{H}^{-1}$)
stretches the posterior error ellipse along that ridge and inflates the
individual marginal error bars ($\Sigma_{ii} > 1 / H_{ii}$).

Every result so far depends on one particular dataset $D$ that we already
collected. Yet if we want to size a benchmark *before* running tests—or prove a
universal accuracy limit across *any* dataset nature might generate—we must
shift from observed curvature ($\mathbf{H}$) to the curvature we expect across
hypothetical future datasets.

---

## 4. The Fisher Information Matrix $\mathcal{I}(\boldsymbol{\theta})$

In 1925, Ronald Fisher asked: *Before we observe a specific dataset $D$, how
much information does an experiment expect to provide on average about
parameter $\boldsymbol{\theta}$?*

To quantify that expected curvature from the random pulls of future data,
Fisher defined the **Score Function** as the gradient of the log-likelihood:

$$
\mathbf{S}(\boldsymbol{\theta}) \equiv \nabla_{\boldsymbol{\theta}} \ln P(D \mid \boldsymbol{\theta})
$$

Intuitively, $\mathbf{S}(\boldsymbol{\theta})$ is the directional "pull" a
dataset exerts on your parameter estimate. Across repeated experiments under the
true $\boldsymbol{\theta}$, random noise pulls equally in all directions so
the pulls cancel out on average ($\mathbb{E}[\mathbf{S}(\boldsymbol{\theta})] =
\mathbf{0}$). However, when the likelihood peak is sharp, small random
fluctuations in the data produce steep slopes and strong pulls—so the variance
of the Score equals the expected negative curvature, defining the **Fisher
Information Matrix $\mathcal{I}(\boldsymbol{\theta})$**:

$$
\mathbf{\mathcal{I}(\boldsymbol{\theta}) \equiv \mathbb{E}_{D \sim P(D \mid \boldsymbol{\theta})} \left[ \mathbf{S}(\boldsymbol{\theta}) \mathbf{S}(\boldsymbol{\theta})^T \right] = \mathbb{E}_{D} \left[ -\nabla_{\boldsymbol{\theta}}^2 \ln P(D \mid \boldsymbol{\theta}) \right]}
$$

> **Observed Hessian vs. Expected Fisher Information**:
> The **Observed Hessian $\mathbf{H}$** is backward-looking: it is computed
> *after* observing one specific dataset $D$ and includes the curvature of the
> Bayesian prior.
> The **Fisher Information $\mathcal{I}(\boldsymbol{\theta})$** is
> forward-looking: it is computed *before* collecting data by averaging the
> likelihood curvature over all possible datasets nature could produce.

---

### 4.1 The Cramér-Rao Lower Bound: The Noise Floor of Estimation

In 1945, Harald Cramér and C.R. Rao proved a foundational bound on estimator
precision:

$$
\mathbf{\text{Cov}(\hat{\boldsymbol{\theta}}) \ge \mathcal{I}(\boldsymbol{\theta})^{-1}}
$$

* **No matter how clever your algorithm is**, no **unbiased estimator**
  $\hat{\boldsymbol{\theta}}$ (an estimator with zero systematic offset on
  average) can achieve an error covariance smaller than
  $\mathcal{I}(\boldsymbol{\theta})^{-1}$.
* The inverse Fisher Information is the **fundamental information-theoretic
  noise floor** of measurement accuracy.
* **Practical Payoff**: The Cramér-Rao Lower Bound (CRLB) tells you **when to
  stop tuning algorithms**. Once an estimator's variance hits
  $\mathcal{I}(\boldsymbol{\theta})^{-1}$, cleverer feature engineering or
  model tweaks cannot reduce uncertainty further without collecting more runs
  ($N$) or cleaner telemetry.

---

### 4.2 The Bridge from Fisher Information to Bayesian Priors: Jeffreys' Prior

Fisher Information $\mathcal{I}(\theta)$ does far more than bound estimator
variance via Cramér-Rao. In 1946, Sir Harold Jeffreys proved that Fisher
Information provides a principled rule for constructing **objective Bayesian
prior distributions that are invariant under any smooth reparameterization**.

#### The Reparameterization Dilemma
Suppose Engineer A parameterizes a flaky test by its failure probability
$\theta \in [0, 1]$ and chooses a "flat" prior $p_\theta(\theta) = 1$.
Meanwhile, Engineer B parameterizes the same physical test by its log-odds:

$$
\phi = \text{logit}(\theta) = \ln \left( \frac{\theta}{1 - \theta} \right)
$$

If Engineer B also chooses a flat prior $p_\phi(\phi) \propto 1$, these two
engineers have made **contradictory physical assumptions**! Transforming
Engineer B's flat prior back to $\theta$ yields:

$$
p(\theta) = p_\phi(\phi) \left| \frac{d\phi}{d\theta} \right| \propto \frac{1}{\theta(1 - \theta)}
$$

which is a strongly non-uniform prior that places infinite density at
$\theta = 0$ and $\theta = 1$.

#### Jeffreys' Invariant Solution
Jeffreys asked: *How do we construct a prior $p(\boldsymbol{\theta})$ such that
transforming coordinates $\boldsymbol{\phi} = g(\boldsymbol{\theta})$ yields
the exact same physical prior?*

By the tensor transformation law of Fisher Information:

$$
\mathcal{I}_\phi(\boldsymbol{\phi}) = \mathbf{J}^T \mathcal{I}_\theta(\boldsymbol{\theta}) \mathbf{J}, \qquad \text{where } \mathbf{J}_{ij} = \frac{\partial \theta_i}{\partial \phi_j}
$$

Taking the determinant and square root on both sides:

$$
\sqrt{\det \mathcal{I}_\phi(\boldsymbol{\phi})} = \sqrt{\det \mathcal{I}_\theta(\boldsymbol{\theta})} \cdot |\det \mathbf{J}|
$$

Intuitively, $\sqrt{\det \mathcal{I}(\boldsymbol{\theta})}$ is the local
**volume-stretching factor** of the coordinate grid—it assigns equal prior mass
per unit of truly distinguishable model behavior, regardless of how coordinates
are stretched. Because this matches the Jacobian change-of-variables rule for
probability densities, defining the prior as:

$$
\mathbf{p_{\text{Jeffreys}}(\boldsymbol{\theta}) \propto \sqrt{\det \mathcal{I}(\boldsymbol{\theta})}}
$$

guarantees complete **coordinate invariance**. No matter how an engineer
parameterizes the model (rate, odds, logit, or **MTBF**, Mean Time Between
Failures), the physical prior is identical.

#### Derivation for a Bernoulli Flaky Test
For a Bernoulli flaky test with failure rate $\theta$, the Fisher Information
is the curvature of binary cross-entropy:

$$
\mathcal{I}(\theta) = \frac{1}{\theta(1 - \theta)}
$$

Taking the square root gives:

$$
p_{\text{Jeffreys}}(\theta) \propto \sqrt{\frac{1}{\theta(1 - \theta)}} = \theta^{-1/2} (1 - \theta)^{-1/2} = \text{Beta}\left(\frac{1}{2}, \; \frac{1}{2}\right)
$$

![Jeffreys Prior Beta(0.5, 0.5) vs. Flat Uniform Beta(1, 1)](../images/ch05_jeffreys_vs_uniform.png)

Why does $\text{Beta}(0.5, 0.5)$ curve upward near $0$ and $1$? Geometrically,
shifting $\theta$ from $0.001 \to 0.011$ (an $11\times$ surge) produces wildly
different pass/fail counts, whereas shifting $0.500 \to 0.510$ looks almost
identical in finite runs. Spreading prior density uniformly in *distinguishable
model behavior* naturally concentrates mass near $0$ and $1$.

**Why Does Jeffreys' Prior Outperform Flat Beta(1, 1) in CI Systems?**
* **Fisher Curvature Weighting**: $\text{Beta}(0.5, 0.5)$ places extra prior
  weight near $\theta \to 0$ and $\theta \to 1$, right where Fisher curvature
  is steepest and a single test outcome carries the most information.
* **No Artificial Inflation on Zero Failures**: If a test passes 10 runs with 0
  failures, a flat prior $\text{Beta}(1, 1)$ produces posterior mean
  $\frac{1}{12} \approx 8.3\%$, falsely flagging a clean test as flaky.
  Jeffreys' prior $\text{Beta}(0.5, 0.5)$ yields posterior mean
  $\frac{0.5}{11} \approx 4.5\%$, cutting false-positive flakiness alerts in
  half on small sample sizes.

Notice what Jeffreys' prior just revealed: raw steps in $\theta$ do not carry
equal informational weight across $[0, 1]$, because the Fisher Information
$\mathcal{I}(\theta)$ itself stretches and compresses the coordinate grid.

---

## 5. Information Geometry: Probability as a Curved Manifold

That coordinate-stretching role of $\mathcal{I}(\boldsymbol{\theta})$ is not
just a trick for choosing priors—it is where Information Theory and
differential geometry meet.

In Euclidean space, the distance between two points $\mathbf{x}$ and
$\mathbf{x} + d\mathbf{x}$ is given by the Pythagorean theorem:

$$
ds^2 = dx_1^2 + dx_2^2 + \dots + dx_n^2
$$

In 1945, C.R. Rao and later Shun'ichi Amari proved that the space of
probability distributions is **not flat Euclidean space**. It is a **curved
Riemannian manifold**!

Think of the metric tensor $g_{ij}(\boldsymbol{\theta})$ as a
position-dependent ruler on a curved globe, where one degree of longitude
shrinks as you approach the poles. To find the ruler separating two neighboring
distributions $P_{\boldsymbol{\theta}}$ and
$P_{\boldsymbol{\theta} + d\boldsymbol{\theta}}$, we Taylor-expand their
Kullback-Leibler divergence around $d\boldsymbol{\theta} = \mathbf{0}$. Both
the 0th-order term ($D_{\text{KL}}(P_{\boldsymbol{\theta}} \parallel
P_{\boldsymbol{\theta}}) = 0$) and the 1st-order gradient term vanish at this
minimum, leaving the purely quadratic Fisher ruler:

$$
D_{\text{KL}}(P_{\boldsymbol{\theta}} \parallel P_{\boldsymbol{\theta} + d\boldsymbol{\theta}}) \approx \frac{1}{2} \sum_{i, j} \mathcal{I}_{ij}(\boldsymbol{\theta}) \, d\theta_i \, d\theta_j
$$

$$
\mathbf{ds^2 = \sum_{i, j} \mathcal{I}_{ij}(\boldsymbol{\theta}) \, d\theta_i \, d\theta_j = 2 \, D_{\text{KL}}(P_{\boldsymbol{\theta}} \parallel P_{\boldsymbol{\theta} + d\boldsymbol{\theta}})}
$$

> **The Geometric Synthesis**: **The Fisher Information Matrix IS the
> Riemannian Metric Tensor of probability space.**
> The distance between two states of nature is measured by how many bits of KL
> divergence separate them.

![Euclidean Parameter Space vs. Fisher-Rao Information Geometry](../images/ch05_info_geometry_ruler.png)

For a Bernoulli test, integrating the local Fisher-Rao ruler
$\int \frac{d\theta}{\sqrt{\theta(1 - \theta)}}$ yields the exact geodesic
arc-length $s(\theta) = 2 \arcsin(\sqrt{\theta})$. Consider two identical
Euclidean steps $\Delta \theta = 0.010$: Step A at the edge ($0.001 \to 0.011$,
an $11\times$ surge in failure rate) spans $\Delta s_A = 0.147\text{ rad}$,
while Step B near the center ($0.500 \to 0.510$, a $1.02\times$ drift) spans
$\Delta s_B = 0.020\text{ rad}$. In information geometry, Step A is
**$7.35\times$ farther apart** ($\Delta s_A / \Delta s_B \approx 7.35$),
capturing the large information difference that Euclidean distance ignores.

### 5.1 Why This Matters in Modern AI: Natural Gradient Descent
In standard gradient descent (e.g. PyTorch `optim.SGD`):

$$
\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \nabla_{\boldsymbol{\theta}} \mathcal{L}
$$

Here $\mathcal{L}$ is the training loss—specifically the **negative
log-likelihood** (or cross-entropy loss) of the model on the observed data.
Standard SGD takes steps in flat Euclidean coordinates: if you change units
(e.g. milliseconds to seconds), the gradient vector bends and slows down. Worse,
near probability boundaries ($p \to 0$ or $p \to 1$), Fisher curvature
$\mathcal{I}$ blows up, so fixed Euclidean steps produce huge, destabilizing KL
divergence jumps.

In **Natural Gradient Descent** (Amari 1998):

$$
\mathbf{\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \, \mathcal{I}(\boldsymbol{\theta})^{-1} \nabla_{\boldsymbol{\theta}} \mathcal{L}}
$$

Multiplying by $\mathcal{I}(\boldsymbol{\theta})^{-1}$ acts as an automatic
curvature brake near steep probability boundaries while accelerating steps
across flat plateaus, ensuring every update moves a controlled distance in
**information divergence** regardless of parameter scaling.

---

## 6. What Do We Actually Do With This Number? (From Curvature to Production Control)

Natural Gradient Descent uses $\mathcal{I}(\boldsymbol{\theta})^{-1}$ to
keep model training stable on a curved manifold. Once a model is fit, those
same curvature tools—the Laplace covariance
$\mathbf{\Sigma} = \mathbf{H}^{-1}$, the Hessian $\mathbf{H}$, and the
Fisher Information $\mathcal{I}(\boldsymbol{\theta})$—become operational
levers for running live systems.

In automated cloud infrastructure and CI orchestration, curvature drives
**three critical actions**:

#### 6.1 Dynamic Timeout Sizing Under Cluster Saturation
In Section 1, our logistic flakiness model revealed that failure probability
rises as worker CPU load ($x_1$), concurrent builds ($x_2$), and network
latency ($x_3$) increase.
* **The Naive Failure**: Setting a fixed timeout (e.g. 5 seconds) causes
  cascading timeout failures whenever cluster load spikes, triggering a
  thundering herd outage.
* **The Curvature Action**: The covariance matrix
  $\mathbf{\Sigma} = \mathbf{H}^{-1}$ gives the exact 95% confidence ellipsoid
  for coefficients $(\beta_1, \beta_2, \beta_3)$, which propagates directly into
  a predictive log-odds variance $\mathbf{x}^T \mathbf{\Sigma} \mathbf{x}$ for
  any telemetry vector $\mathbf{x}$. When cluster CPU utilization spikes to
  $85\%$, the CI orchestrator uses the **upper boundary of this predictive
  curvature interval** to dynamically scale test timeout thresholds, preventing
  spurious test failures without masking genuine application deadlocks.

#### 6.2 Automatic Detection of Redundant Telemetry (Multicollinearity)
In modern observability stacks, engineers collect thousands of metrics (CPU
load, thread counts, context switches, memory pressure).
* If two metrics measure the same physical bottleneck, the log-posterior
  surface becomes a flat, elongated ridge: the second derivative along that
  ridge drops toward zero.
* **The Curvature Action**: The monitoring bot computes the eigenvalues of the
  Hessian $\mathbf{H}$, where $\lambda_{\max}$ measures the steep wall *across*
  the ridge and $\lambda_{\min}$ measures the flat spine *along* it. If the
  condition number $\kappa = \frac{\lambda_{\max}}{\lambda_{\min}} > 10^4$,
  the system flags **severe multicollinearity** and prunes redundant telemetry
  streams, reducing time-series database storage costs by up to $60\%$ while
  stabilizing regression solvers.

#### 6.3 Sizing Calibration Benchmarks via the Cramér-Rao Bound
Before deploying a new CI worker fleet (e.g. migrating from Intel to ARM
Graviton processors), SREs must benchmark test execution stability.
* How many benchmark runs are required to calibrate the new environment?
* **The Curvature Action**: Because Fisher information adds linearly across
  independent runs ($\mathcal{I}_N = N \cdot \mathcal{I}_1$), the Cramér-Rao
  Lower Bound gives a minimum standard error
  $\text{SE}(\hat{\beta}_i) \ge \sqrt{(\mathcal{I}_1^{-1})_{ii} / N}$. If we
  require the flakiness coefficient $\beta_{\text{CPU}}$ to be calibrated
  within a $\pm 0.10$ margin of error, we invert the single-run Fisher
  Information to compute the **exact minimum benchmark sample size
  $N_{\text{benchmark}}$** required before the new fleet can safely handle
  production builds.

Across timeout sizing, telemetry pruning, and fleet benchmarking, every
decision reduces to measuring how steeply log-probability bends around your
operating point.

---

## 7. Summary: The Golden Rules of Chapter 05

Keep these six geometric principles in mind whenever you quantify uncertainty
or optimize models:

1. **The Laplace approximation turns complex posteriors into Gaussians**:
   Around the MAP mode, $\ln P(\boldsymbol{\theta} \mid D)$ is locally
   quadratic.
2. **The inverse Hessian is the covariance of belief**:
   $\mathbf{\Sigma} = \mathbf{H}^{-1}$. Steep curvature equals tight certainty;
   flat curvature equals high uncertainty.
3. **The Fisher Information Matrix is expected curvature**:
   $\mathcal{I}(\boldsymbol{\theta}) = \mathbb{E}[-\nabla^2 \ln P]$ is the
   expected information content of an experiment before data arrives.
4. **The Cramér-Rao bound is the information-theoretic noise floor**: No
   unbiased estimator can achieve covariance lower than
   $\mathcal{I}(\boldsymbol{\theta})^{-1}$.
5. **The Fisher Matrix is the metric tensor of probability**: Information space
   is curved, and the true distance between two distributions is measured by KL
   divergence: $ds^2 = d\boldsymbol{\theta}^T \mathcal{I} d\boldsymbol{\theta}$.
6. **Curvature guides objective priors and invariant optimization**: **Jeffreys'
   Prior** ($p(\boldsymbol{\theta}) \propto \sqrt{\det \mathcal{I}}$) keeps
   Bayesian priors invariant across reparameterizations, while **Natural
   Gradient Descent** ($\mathcal{I}^{-1}\nabla\mathcal{L}$) keeps optimization
   steps uniform in information divergence.

Now, open the companion Python Lab to fit a multi-parameter logistic flakiness
model, compute the numerical Hessian, and plot the curved information
ellipses!

---

**[Course Home](../README.md) | [Python Lab](../python/05_curvature_laplace_and_fisher_information.ipynb) | [Prev: Ch 04: Shared Outages & Venn Geometry](04_mutual_information_and_venn_geometry.ipynb) | [Next: Ch 06: The Engine of Computation: MCMC](06_mcmc_mechanics_and_entropy.ipynb)**
